# Bronze Layer - Ingesta con Auto Loader

Ingesta automática de archivos CSV desde el volume `e_commerce.landing_zone.raw_data` hacia la tabla Delta `e_commerce.bronze.bronze_pedidos_raw` usando **Auto Loader** (`cloudFiles`) con batch-streaming idempotente.

## Características

* **cloudFiles**: Auto Loader detecta automáticamente archivos nuevos en el directorio
* **Trigger.AvailableNow**: procesa todos los archivos pendientes y se detiene (no streaming continuo)
* **Checkpoint**: evita reprocesar archivos ya ingeridos (idempotencia)
* **Schema evolution**: `addNewColumns` detecta y agrega columnas nuevas automáticamente
* **`_ingested_at`**: timestamp de auditoría para saber cuándo se ingeró cada registro

In [0]:
from pyspark.sql.functions import current_timestamp, col

SOURCE_PATH = "/Volumes/e_commerce/landing_zone/raw_data"
CHECKPOINT_PATH = "/Volumes/e_commerce/bronze/checkpoints/bronze_pedidos_raw"
SCHEMA_LOCATION = "/Volumes/e_commerce/bronze/checkpoints/schemas/bronze_pedidos_raw"
TABLE_NAME = "e_commerce.bronze.bronze_pedidos_raw"

INITIAL_LOAD = True

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS e_commerce.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS e_commerce.bronze.checkpoints")

DataFrame[]

In [0]:
def ingest_bronze(source_path, checkpoint_path, schema_location, table_name):
    """
    Ingesta CSV con Auto Loader (cloudFiles) + Trigger.AvailableNow.

    - cloudFiles: detecta automáticamente archivos nuevos en el directorio
    - Trigger.AvailableNow: procesa todos los archivos pendientes y se detiene
    - Checkpoint: evita reprocesar archivos ya ingeridos (idempotencia)
    - Schema evolution: addNewColumns detecta columnas nuevas
    - _ingested_at: timestamp de auditoría
    """
    try:
        count_before = spark.table(table_name).count()
    except Exception:
        count_before = 0

    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "false")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .load(source_path)
        .withColumn("_ingested_at", current_timestamp())
        .writeStream
        .format("delta")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .toTable(table_name)
        .awaitTermination()
    )

    count_after = spark.table(table_name).count()
    print(f"Ingesta completada: {table_name}")
    print(f"  Registros antes:  {count_before}")
    print(f"  Registros después: {count_after}")
    print(f"  Nuevos registros:  {count_after - count_before}")
    return count_after

In [0]:
if INITIAL_LOAD:
    print("Modo carga inicial: eliminando tabla y checkpoint existentes...")
    spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")
    try:
        dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
    except Exception:
        pass
    try:
        dbutils.fs.rm(SCHEMA_LOCATION, recurse=True)
    except Exception:
        pass

total = ingest_bronze(
    source_path=SOURCE_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    schema_location=SCHEMA_LOCATION,
    table_name=TABLE_NAME,
)

Modo carga inicial: eliminando tabla y checkpoint existentes...
Ingesta completada: e_commerce.bronze.bronze_pedidos_raw
  Registros antes:  0
  Registros después: 15750
  Nuevos registros:  15750


In [0]:
print(f"=== Verificación: {TABLE_NAME} ===")
df = spark.table(TABLE_NAME)
print(f"Total registros: {df.count()}")
print(f"\nSchema:")
df.printSchema()
print(f"\nMuestra (5 filas):")
display(df.limit(5))

print("\n=== Test de idempotencia ===")
total_replay = ingest_bronze(
    source_path=SOURCE_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    schema_location=SCHEMA_LOCATION,
    table_name=TABLE_NAME,
)

=== Verificación: e_commerce.bronze.bronze_pedidos_raw ===
Total registros: 15750

Schema:
root
 |-- id_transaccion: string (nullable = true)
 |-- fecha_operacion: string (nullable = true)
 |-- cliente_info: string (nullable = true)
 |-- productos_comprados: string (nullable = true)
 |-- monto_total: string (nullable = true)
 |-- estado_pago: string (nullable = true)
 |-- direccion_envio: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)


Muestra (5 filas):


id_transaccion,fecha_operacion,cliente_info,productos_comprados,monto_total,estado_pago,direccion_envio,_rescued_data,_ingested_at
TRX-011820,07/05/2025,Valentina Ramirez Cardoso|valentina_ramirez_cardoso19@email.com|Rosario,Termo Stanley 1.4L (x3),"$255,000.00",COMPLETADO,Belgrano 3632,null,2026-10-12T18:37:55.269Z
TRX-001211,Semana pasada,Agustin Morales Gimenez|agustin_morales_gimenez32@email.com|Buenos Aires,Cargador Portatil 10000mAh (x2) / Teclado Mecanico RGB (x1) / Cinto de Cuero (x1),USD 104.2,Pendiente,Rivadavia 1166,null,2026-10-12T18:37:55.269Z
TRX-009467,14/10/2025,Joaquin Acosta Mendoza|joaquin_acosta_mendoza30@email.com|Salta,Cafetera Italiana 6 Tazas (x3) / Mouse Inalambrico Ergonomico (x3),"ARS $ 274.500,00",en_proceso,Belgrano 3271,null,2026-10-12T18:37:55.269Z
TRX-004338,2026-08-24 09:02:00,Camila Benitez Peralta|camila_benitez_peralta25@email.com|Rosario,"Gorra Trucker (x3), Teclado Mecanico RGB (x2)",155000,Rechazado,null,null,2026-10-12T18:37:55.269Z
TRX-012468,15-Jan-2025,Lucia Diaz Silva|lucia_diaz_silva23@email.com|Mendoza,Cafetera Italiana 6 Tazas (x3),216000,Pagado,Belgrano 4658,null,2026-10-12T18:37:55.269Z



=== Test de idempotencia ===
Ingesta completada: e_commerce.bronze.bronze_pedidos_raw
  Registros antes:  15750
  Registros después: 15750
  Nuevos registros:  0
✅ Idempotencia verificada: no se duplicaron registros
